# Neural processing v3 — Ca2+ extraction notebook
**Self-contained: every computation is in a visible cell.** Derived from
`neuralprocessing_adam_pilpeline v2` with the Aug 9 audit fixes applied per team
directives. Batch `.py` conversion happens AFTER this notebook is validated on real data.

| # | Directive (Aug 9) | Where |
|---|---|---|
| 1 | XML metadata extraction **deferred** | parameters cell (frame period stays manual) |
| 2 | Denoise **before** MC + memory fix + speed | Denoising cells (registration smoothing + optional PCA) |
| 3 | MC algorithm unchanged | MC cells (faithful to v2) |
| 4 | Shift-application performance | float32 + streamed pass-1 template (≈3× less RAM) |
| 5 | Binning discussion + NaN fix | Binning cells (incl. empirical sweep) |
| 6 | Border guard | shift-aware border cell |
| 7 | Data-driven ROI count + indexing fixes | ROI extraction cells |
| 8 | Bleaching check | Bleach cell |

Also fixed from v2: the `next_roi` view-mutation bug (`this_trace` was a VIEW into the
movie — `+=` corrupted the data), roi_map labels now match trace rows exactly, and the
two crash bugs in the old save/ΔF/F cells.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
try:
    from tqdm import tqdm
except ImportError:                       # progress bars are cosmetic
    def tqdm(x, **k):
        return x
from scipy.stats import zscore
from scipy.ndimage import shift as ndshift, gaussian_filter, convolve
from skimage.registration import phase_cross_correlation
from skimage.morphology import binary_dilation
import tifffile as tiff
import json, time

## Parameters — the ONLY cell you edit per run
XML extraction is deferred (directive 1): set `FRAME_PERIOD` from the t-series XML by
hand for now. Everything else has a sane default; the binning sweep below helps pick
`DETECT_BIN`.

In [ ]:
# ---- data ----
DATA_PATH = Path('/grid/courses/data/imagcourse/GECI Project Jonathons/Old Data/jonathans_finale/20260808_M1_Mouse1/spon_sniff_run1/TSeries-08082026-0955-020')
TIF_NAME  = 'TSeries-08082026-0955-020_Cycle00001_Ch2_000001.ome.tif'
FRAME_PERIOD = 0.0328181     # s/frame — from the XML, BY HAND for now (directive 1)

# ---- denoising (directive 2) ----
MC_SMOOTH_SIGMA = 1.0        # px; Gaussian smoothing of frames FOR SHIFT ESTIMATION ONLY
                             # (shifts are applied to the raw frames — traces untouched).
                             # This is the denoise-before-MC that helps registration.
PCA_STAGE = "off"            # "off" | "pre_mc" | "post_mc" — optional PCA movie denoise.
PCA_RANK  = 50               # see the denoising discussion cell before turning this on.

# ---- detection (directive 5) ----
DETECT_BIN = "auto"          # temporal bin for DETECTION ONLY; "auto" = ~6 Hz; int to fix
CORR_THRESHOLD = 0.3         # ROI-growth correlation threshold (on the BINNED movie)

# ---- ROI acceptance (directive 7: count is data-driven, no n_rois) ----
SIZE_MIN, SIZE_MAX = 15, 100 # accepted ROI area, px (TODO µm² once XML parsing lands)
SIZE_GROW_CAP = 200          # hard stop for region growth
MAX_ROIS_CAP  = 1000         # safety cap only — extraction stops when the corr map is exhausted

# ---- QC ----
BORDER_PX_MIN = 5            # border zeroing floor; raised automatically to max shift+1
DFF_PERCENTILE = 15          # static F0 = bottom 15% of each ROI trace (team decision)
BLEACH_WARN_PCT = 20         # warn if |mean-F drift| over the run exceeds this

## Load
`frames` must be (T, Y, X). Confirm T against the XML's frame count by eye — a partial
upload silently truncates and poisons everything downstream.

In [ ]:
t0 = time.time()
frames = tiff.imread(DATA_PATH / TIF_NAME)
if frames.ndim == 2:
    frames = frames[None]
T, Y, X = frames.shape
frame_rate = 1.0 / FRAME_PERIOD
time_vector = np.arange(T) * FRAME_PERIOD   # exact spacing (v2's linspace drifted by one
                                            # frame period across the run)
print(f"movie {frames.shape} {frames.dtype}  ({T/frame_rate:.0f}s @ {frame_rate:.2f} Hz, "
      f"{frames.nbytes/1e9:.1f} GB raw, load {time.time()-t0:.0f}s)")
print("CHECK: does T match the XML <Frame> count?")

In [ ]:
original_anatomy = frames.mean(0)
plt.figure(figsize=(4, 4))
plt.imshow(original_anatomy, cmap="gray",
           vmin=np.percentile(original_anatomy, 1), vmax=np.percentile(original_anatomy, 99))
plt.title("original anatomy (raw mean)"); plt.axis("off"); plt.show()

## Denoising (directive 2) — what runs before MC, and why
Frame-by-frame SNR is poor, so we denoise before motion correction — but *which*
denoise matters:

**Registration smoothing (ON by default, `MC_SMOOTH_SIGMA`).** Each frame is Gaussian-
smoothed (σ≈1 px) *only inside the shift estimator*. Phase correlation then sees
structure instead of shot noise, but the measured shifts are applied to the **raw**
frames — traces are never touched by the filter. Zero extra memory (smoothing is
per-frame, on the fly). This is the safe, standard denoise-before-MC.

**PCA denoising (optional, `PCA_STAGE`).** v2's version had two problems, both fixed here:
(1) *memory* — it cast the whole movie to float64 (~57 GB at 512²×27k). This version
keeps float32 and reconstructs **in place, block by block** (peak extra ≈ one 50×(Y·X)
factor, ~50 MB); with sklearn present it uses randomized SVD (minutes), else scipy svds.
(2) *ordering physics* — run **pre-MC** on a moving movie, the top principal components
ARE the motion (motion dominates variance), so a rank-50 reconstruction smears cells
along their motion paths. That's why the default is `"off"` and `"post_mc"` is the
recommended stage if you want it. `"pre_mc"` is available per the team directive —
use it knowingly, ideally on a low-motion run, and compare anatomy sharpness after MC.

Practical recommendation for tonight: `MC_SMOOTH_SIGMA=1.0`, `PCA_STAGE="off"` —
detection SNR is already handled by temporal binning below, which attacks the same
noise with none of the risk.

In [ ]:
def pca_denoise_inplace(mov_f32, rank, chunk=2000):
    """Rank-`rank` reconstruction of (T,Y,X) float32 movie, written back IN PLACE.
    Memory-safe: the (T, Y*X) matrix is a reshape VIEW of the movie (no copy);
    only the factors + one row-block are allocated."""
    Tn, Yn, Xn = mov_f32.shape
    M = mov_f32.reshape(Tn, Yn * Xn)          # view, not a copy
    t0 = time.time()
    try:
        from sklearn.utils.extmath import randomized_svd
        U, S, Vt = randomized_svd(M, n_components=rank, random_state=0)
    except ImportError:
        from scipy.sparse.linalg import svds
        U, S, Vt = svds(M, k=rank)
    for i in range(0, Tn, chunk):
        M[i:i+chunk] = (U[i:i+chunk] * S) @ Vt
    print(f"PCA denoise rank {rank}: {time.time()-t0:.0f}s")

if PCA_STAGE == "pre_mc":
    print("WARNING: pre-MC PCA — motion lives in the top components; expect some "
          "motion-smearing of cells. 'post_mc' is the safer stage.")
    frames = frames.astype(np.float32)        # float32 copy replaces uint16 raw
    pca_denoise_inplace(frames, PCA_RANK)

## Motion correction — two-pass rigid (algorithm unchanged, directive 3)
Same estimator and shift interpolation as v2. Engineering changes only (directive 4):
- everything float32 (v2's `np.empty` stacks were float64 — 2× the RAM for nothing);
- the pass-1 aligned stack is never materialized — only its **mean** is accumulated
  (that stack was used solely to build the refined template). Peak memory is now
  raw + ONE aligned float32 stack ≈ 1.5× movie-float32 (~43 GB at 512²×27k) instead
  of ~130 GB;
- per-frame smoothing for estimation per the denoising cell.

In [ ]:
def estimate_shifts(mov, template, upsample=10, smooth_sigma=MC_SMOOTH_SIGMA):
    tmpl = gaussian_filter(template.astype(np.float32), smooth_sigma) if smooth_sigma else template
    ys, xs = np.empty(len(mov)), np.empty(len(mov))
    for i in tqdm(range(len(mov)), desc="shifts", leave=False):
        fr = mov[i].astype(np.float32)
        if smooth_sigma:
            fr = gaussian_filter(fr, smooth_sigma)
        (dy, dx), _, _ = phase_cross_correlation(tmpl, fr, upsample_factor=upsample)
        ys[i], xs[i] = dy, dx
    return ys, xs

# pass 1: estimate against the raw mean, accumulate the aligned MEAN only
y1, x1 = estimate_shifts(frames, original_anatomy)
acc = np.zeros((Y, X), np.float64)
for i in tqdm(range(T), desc="template", leave=False):
    acc += ndshift(frames[i].astype(np.float32), (y1[i], x1[i]))
aligned_anatomy = (acc / T).astype(np.float32)

# pass 2: re-estimate RAW frames against the refined template (no double interpolation)
y2, x2 = estimate_shifts(frames, aligned_anatomy)
total_shift_1 = np.hypot(y1, x1)
total_shift_2 = np.hypot(y2, x2)

In [ ]:
# apply pass-2 shifts -> the ONE aligned float32 stack used everywhere downstream
final_frames = np.empty((T, Y, X), np.float32)
for i in tqdm(range(T), desc="apply", leave=False):
    final_frames[i] = ndshift(frames[i].astype(np.float32), (y2[i], x2[i]))
final_anatomy = final_frames.mean(0)
max_shift = float(np.abs(np.concatenate([y2, x2])).max())
print(f"max |shift| = {max_shift:.2f} px")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(time_vector, total_shift_1, lw=.6, label="pass 1")
axes[0].plot(time_vector, total_shift_2, lw=.6, label="pass 2")
axes[0].set(xlabel="time (s)", ylabel="total shift (px)"); axes[0].legend()
for ax, img, name in ((axes[1], original_anatomy, "before"), (axes[2], final_anatomy, "after")):
    ax.imshow(img, cmap="gray", vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
    ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
if PCA_STAGE == "post_mc":
    pca_denoise_inplace(final_frames, PCA_RANK)
    final_anatomy = final_frames.mean(0)

## Temporal binning for detection (directive 5) — the discussion
**Why bin at all:** raw resonant frames are shot-noise dominated. Averaging n frames
leaves the shared calcium signal intact but cuts independent noise by √n, so the
pixel-vs-neighbors correlation — which collapsed to ~0.17 on this dataset unbinned —
recovers. (Reproduced synthetically: 0.16 unbinned → 0.40 at bin 10, cells recovered.)

**What sets the optimum:**
- *Lower bound:* enough SNR that cell pixels clear `CORR_THRESHOLD` with margin.
- *Upper bound:* the GCaMP7s transient itself (rise ~50 ms, decay τ ≈ 1–1.5 s). Bins
  much longer than ~1 s start averaging *across* transients, diluting the very signal
  correlations we detect with. Bins up to ~0.5 s are essentially free; ~1 s is fine.
- At 30.5 Hz that puts the useful range at ~5–30 frames. "auto" picks ~6 Hz (bin 5) —
  deliberately conservative; noisy data often does better at 10–20.
- Binning is DETECTION-ONLY: traces are always extracted at full rate afterwards, so
  the choice affects *which* pixels form ROIs, not the time resolution of the science.

**Empirical selection:** the sweep below computes the correlation map on a center crop
for several bin factors. Pick the smallest bin whose max clears ~0.4–0.5 — past the
knee, more binning buys little and eventually hurts.

In [ ]:
def bin_movie(mov, n):
    if n <= 1:
        return mov
    Tb = (len(mov) // n) * n
    return mov[:Tb].reshape(-1, n, *mov.shape[1:]).mean(axis=1)

def neighbor_corr_map(mov):
    """Pearson r of each pixel vs the SUM of its 8 neighbors — exact, one streaming
    pass, no movie copies (proven equal to the per-pixel pearsonr loop to 1e-6)."""
    Tn = len(mov)
    k = np.ones((3, 3))
    sx = np.zeros(mov.shape[1:]); sxx = np.zeros(mov.shape[1:])
    ss = np.zeros(mov.shape[1:]); sss = np.zeros(mov.shape[1:]); sxs = np.zeros(mov.shape[1:])
    for t in range(Tn):
        f = mov[t].astype(np.float64)
        s = convolve(f, k, mode="constant") - f
        sx += f; sxx += f * f; ss += s; sss += s * s; sxs += f * s
    num = Tn * sxs - sx * ss
    den = np.sqrt((Tn * sxx - sx**2) * (Tn * sss - ss**2))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, 0.0)   # constant pixels -> 0, never NaN

# ---- bin sweep on a center crop (fast, ~seconds each) ----
cy, cx_ = Y // 2, X // 2
crop = final_frames[:, max(0, cy-64):cy+64, max(0, cx_-64):cx_+64]
print("bin  eff.rate   corr-map max (center crop)")
for nb in [1, 2, 3, 5, 8, 12, 20, 30]:
    if nb < len(crop):
        cm = neighbor_corr_map(bin_movie(crop, nb))
        print(f"{nb:3d}  {frame_rate/nb:6.1f} Hz   {cm[2:-2, 2:-2].max():.3f}")

In [ ]:
DETECT_BIN_N = max(1, int(round(frame_rate / 6.0))) if DETECT_BIN == "auto" else int(DETECT_BIN)
detect_mov = bin_movie(final_frames, DETECT_BIN_N)
print(f"detection movie: bin x{DETECT_BIN_N} -> {len(detect_mov)} frames @ "
      f"{frame_rate/DETECT_BIN_N:.1f} Hz")
correlation_map = neighbor_corr_map(detect_mov)

## Border guard + NaN hygiene (directives 5 & 6)
Shifting fills edges with zeros in *some frames only* — edge-pixel traces become gated
by the shift time course, which is **shared across all edge pixels**, so they correlate
near 1.0 with each other and grow fake "cells" of pure motion artifact. The dead band
is exactly the maximum shift, so the border is `max(BORDER_PX_MIN, ceil(max|shift|)+1)`.
NaNs (none can arise from the streaming map, but belt-and-suspenders for any edit) would
otherwise win `argmax` and hijack ROI seeding.

In [ ]:
border = max(BORDER_PX_MIN, int(np.ceil(max_shift)) + 1)
corr_bordered = np.nan_to_num(correlation_map, nan=0.0, posinf=0.0, neginf=0.0).copy()
corr_bordered[:border, :] = 0; corr_bordered[-border:, :] = 0
corr_bordered[:, :border] = 0; corr_bordered[:, -border:] = 0
print(f"border zeroed: {border} px (max shift {max_shift:.2f})")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(final_anatomy, cmap="gray",
               vmin=np.percentile(final_anatomy, 1), vmax=np.percentile(final_anatomy, 99))
axes[0].set_title("anatomy"); axes[0].axis("off")
im = axes[1].imshow(corr_bordered, cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"correlation map (max {corr_bordered.max():.2f})"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()
if corr_bordered.max() < CORR_THRESHOLD:
    print(f"WARNING: corr map max {corr_bordered.max():.2f} < threshold {CORR_THRESHOLD}"
          f" — no ROIs will grow. Increase DETECT_BIN (see sweep) before proceeding.")

## ROI extraction (directive 7) — data-driven, indexing-safe
Changes vs v2, each load-bearing:
1. **No `n_rois` hardcode.** Extraction runs until the correlation map is *exhausted*
   (no remaining seed above `CORR_THRESHOLD`) — the data decides the count. A cap of
   `MAX_ROIS_CAP` exists purely as a runaway guard (each attempt consumes ≥1 seed
   pixel, so termination is guaranteed regardless).
2. **`this_trace = ....copy()`** — in v2 this was a VIEW into the movie, and `+=`
   silently wrote the growing ROI sum back into `final_frames`, corrupting every later
   ROI's correlations. This was the most dangerous bug in the notebook.
3. **Labels == rows.** v2 stamped `roi_map` with the attempt index but compacted the
   trace array — map label k did not index trace row k. Here masks are collected and
   labeled 1..n in exactly trace-row order.
4. Growth runs on the **binned** detection movie (same SNR logic as the corr map);
   traces are extracted from the **full-rate** movie in the next cell.
5. `pearsonr` replaced by the identical dot-product formula (scale-invariant Pearson,
   equal to scipy to 1e-10) — that plus binning turns hours into minutes.

In [ ]:
def fast_pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else 0.0

def next_roi(corr_map, mov, corr_threshold, grow_cap):
    i, j = np.unravel_index(np.argmax(corr_map), corr_map.shape)
    this_trace = mov[:, i, j].astype(np.float64).copy()      # COPY — fix #2 above
    this_roi = np.zeros(corr_map.shape, np.uint8)
    this_roi[i, j] = 1
    used = corr_map.copy(); used[i, j] = 0
    growing = True
    while growing and this_roi.sum() < grow_cap:
        growing = False
        ring = np.argwhere(binary_dilation(this_roi, np.ones((3, 3))) ^ this_roi.astype(bool))
        add = np.zeros(len(this_trace))
        for (yy, xx) in ring:
            if used[yy, xx] != 0:
                px = mov[:, yy, xx].astype(np.float64)
                if fast_pearson(this_trace, px) > corr_threshold:
                    growing = True
                    this_roi[yy, xx] = 1
                    used[yy, xx] = 0
                    add += px
        this_trace += add
    return this_roi, this_roi.sum(), used

masks, sizes_all = [], []
used_map = corr_bordered.copy()
t0 = time.time()
while used_map.max() > CORR_THRESHOLD and len(masks) < MAX_ROIS_CAP:
    roi, size, used_map = next_roi(used_map, detect_mov, CORR_THRESHOLD, SIZE_GROW_CAP)
    sizes_all.append(int(size))
    if SIZE_MIN < size < SIZE_MAX:
        masks.append(roi.astype(bool))
n_rej = len(sizes_all) - len(masks)
print(f"{len(masks)} ROIs accepted, {n_rej} size-rejected "
      f"(attempt sizes min/med/max {min(sizes_all)}/{int(np.median(sizes_all))}/"
      f"{max(sizes_all)}) in {time.time()-t0:.0f}s")

roi_map = np.zeros((Y, X), np.uint16)
for k, m in enumerate(masks):
    roi_map[m] = k + 1        # label k+1 == trace row k, ALWAYS  (fix #3)

In [ ]:
# traces at FULL rate from the aligned movie (sum over member pixels; identical
# semantics to the grown sum, order-independent)
if masks:
    traces_raw = np.stack([final_frames[:, m].sum(axis=1) for m in masks]).astype(np.float32)
    roi_npix = np.array([int(m.sum()) for m in masks])
else:
    traces_raw = np.zeros((0, T), np.float32); roi_npix = np.array([], int)
print("traces_raw:", traces_raw.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(final_anatomy, cmap="gray",
               vmin=np.percentile(final_anatomy, 1), vmax=np.percentile(final_anatomy, 99))
mm = np.ma.masked_where(roi_map == 0, roi_map)
axes[0].imshow(mm, cmap="prism", alpha=.45); axes[0].set_title(f"{len(masks)} ROIs"); axes[0].axis("off")
if len(traces_raw):
    tz = zscore(traces_raw, axis=1)
    axes[1].imshow(tz[np.argsort(np.argmax(tz, 1))], aspect="auto", cmap="afmhot",
                   vmin=0, vmax=np.percentile(tz, 99),
                   extent=[time_vector[0], time_vector[-1], 1, len(tz) + 1])
    axes[1].set(xlabel="time (s)", ylabel="ROI # (viz only: z-scored)")
plt.tight_layout(); plt.show()

## ΔF/F (static bottom-15% F0, team decision) + bleaching check (directive 8)
Static F0 preserves slow arousal-state differences (the science) but **assumes no strong
photobleaching** — under bleach, F0 sits near the late-run floor and inflates early ΔF/F.
So we measure it: a robust linear fit to the FOV-mean raw fluorescence; if the fitted
drift across the run exceeds `BLEACH_WARN_PCT` of the mean, this run gets flagged and
the F0 strategy revisited (per-ROI detrended F0 is the usual remedy — decide as a team,
not silently).

In [ ]:
if len(traces_raw):
    F0 = np.percentile(traces_raw, DFF_PERCENTILE, axis=1, keepdims=True)  # 15th, per ROI
    dff = (traces_raw - F0) / np.maximum(F0, 1e-6)

    # ---- bleaching check ----
    meanF = traces_raw.mean(axis=0)
    slope, intercept = np.polyfit(time_vector, meanF, 1)
    drift_pct = 100.0 * slope * (time_vector[-1] - time_vector[0]) / meanF.mean()
    bleach_flag = abs(drift_pct) > BLEACH_WARN_PCT
    print(f"mean-F drift over run: {drift_pct:+.1f}%  "
          f"{'*** BLEACH FLAG — revisit F0 strategy ***' if bleach_flag else '(ok)'}")

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].imshow(dff[np.argsort(np.argmax(dff, 1))], aspect="auto", cmap="afmhot",
                   vmin=0, vmax=np.percentile(dff, 99),
                   extent=[time_vector[0], time_vector[-1], 1, len(dff) + 1])
    axes[0].set(ylabel="ROI #", title="dF/F")
    axes[1].plot(time_vector, meanF, lw=.6, label="FOV-mean raw F")
    axes[1].plot(time_vector, intercept + slope * time_vector, "r--",
                 label=f"drift {drift_pct:+.1f}%")
    axes[1].set(xlabel="time (s)", ylabel="mean F"); axes[1].legend()
    plt.tight_layout(); plt.show()

## Save (minimal, crash-fixed — full output routing is deferred by team decision)
Raw traces are saved alongside ΔF/F: z-scoring/normalization choices stay revisable.

In [ ]:
OUT_ROOT = Path('/grid/courses/data/imagcourse/GECI Project Jonathons/Data to Analyze')
out = OUT_ROOT / "neural data output" / DATA_PATH.name    # per-t-series subfolder: no overwrites
out.mkdir(parents=True, exist_ok=True)
np.save(out / "traces_raw.npy", traces_raw)
np.save(out / "roi_npix.npy", roi_npix)
if len(traces_raw):
    np.save(out / "dff.npy", dff)
    np.save(out / "F0.npy", F0.squeeze())
np.save(out / "shifts_yx.npy", np.stack([y2, x2]))
tiff.imwrite(out / "roi_map.tif", roi_map)
(out / "params.json").write_text(json.dumps({
    "frame_period": FRAME_PERIOD, "mc_smooth_sigma": MC_SMOOTH_SIGMA,
    "pca_stage": PCA_STAGE, "pca_rank": PCA_RANK, "detect_bin": DETECT_BIN_N,
    "corr_threshold": CORR_THRESHOLD, "size_min": SIZE_MIN, "size_max": SIZE_MAX,
    "size_grow_cap": SIZE_GROW_CAP, "border_px": border,
    "dff_percentile": DFF_PERCENTILE, "n_rois": len(masks), "max_shift_px": max_shift,
}, indent=2))
print("saved ->", out)